# California County Climate Report

Generates a one-page, report-style climate summary for any of California's
58 counties using `climakitae.ClimateData().get()` and the reusable
rendering components in `climakitae.visualize`.

**What you get**

| Panel | Description |
|---|---|
| Stat cards | Three headline numbers (avg-high rise, heat-wave frequency, extra hot days) |
| Summary table | Metric × period table: Historical · GWL 1.5 °C · GWL 2.0 °C |
| Bar chart | Countywide 1-in-10-yr extreme heat threshold: historical vs projection |

**Data**: WRF dynamical downscaling — daily `t2max` / `t2min`, 9 km grid
(`d02`), converted to °F, clipped to the selected county.

**Periods**

| Period | Definition |
|---|---|
| Historic (1981–2010) | 30-yr reference window from WRF historical runs |
| GWL 1.5 °C | ±15 yr window centred on each model's 1.5 °C crossing |
| GWL 2.0 °C | ±15 yr window centred on each model's 2.0 °C crossing |

> **Caveat** — The 1-in-10-yr threshold is an annual-max-quantile estimate.
> For rigorous return-period analysis, use the `metric_calc one_in_x`
> processor (GEV fit), preferably on LOCA2 for better tail estimates.
> WRF ensemble size is smaller than LOCA2; treat summaries as central
> tendencies, not the full uncertainty range.

## 0 · Install `climakitae.visualize`

Until [cal-adapt/climakitae#765](https://github.com/cal-adapt/climakitae/pull/765)
is merged, install directly from the feature branch:

In [ ]:
!uv pip install -q 'climakitae @ git+https://github.com/cal-adapt/climakitae.git@feature/report-visualize'

## 1 · Imports

In [ ]:
from __future__ import annotations

import warnings

import ipywidgets as widgets
import pandas as pd
from IPython.display import display

from climakitae.new_core.user_interface import ClimateData
from climakitae.visualize import (
    PeriodInputs,
    build_report_figure,
    compute_report_metrics,
)

warnings.filterwarnings("ignore", category=FutureWarning)

## 2 · Choose county and thresholds

In [ ]:
CA_COUNTIES = [
    "Alameda County", "Alpine County", "Amador County", "Butte County",
    "Calaveras County", "Colusa County", "Contra Costa County", "Del Norte County",
    "El Dorado County", "Fresno County", "Glenn County", "Humboldt County",
    "Imperial County", "Inyo County", "Kern County", "Kings County",
    "Lake County", "Lassen County", "Los Angeles County", "Madera County",
    "Marin County", "Mariposa County", "Mendocino County", "Merced County",
    "Modoc County", "Mono County", "Monterey County", "Napa County",
    "Nevada County", "Orange County", "Placer County", "Plumas County",
    "Riverside County", "Sacramento County", "San Benito County",
    "San Bernardino County", "San Diego County", "San Francisco County",
    "San Joaquin County", "San Luis Obispo County", "San Mateo County",
    "Santa Barbara County", "Santa Clara County", "Santa Cruz County",
    "Shasta County", "Sierra County", "Siskiyou County", "Solano County",
    "Sonoma County", "Stanislaus County", "Sutter County", "Tehama County",
    "Trinity County", "Tulare County", "Tuolumne County", "Ventura County",
    "Yolo County", "Yuba County",
]

county_w = widgets.Dropdown(
    options=CA_COUNTIES,
    value="Los Angeles County",
    description="County:",
    layout=widgets.Layout(width="380px"),
    style={"description_width": "initial"},
)
hot_day_w = widgets.IntSlider(
    value=90, min=80, max=115, step=1,
    description="Hot-day threshold (\u00b0F):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)
heatwave_w = widgets.IntSlider(
    value=4, min=2, max=10, step=1,
    description="Heat-wave min consecutive days:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)
gwl_w = widgets.SelectMultiple(
    options=[1.5, 2.0, 2.5, 3.0, 4.0],
    value=(1.5, 2.0),
    description="Warming levels (\u00b0C):",
    style={"description_width": "initial"},
    rows=5,
)

display(widgets.VBox([
    widgets.HTML("<b>Select county and thresholds, then run the cells below.</b>"),
    county_w,
    hot_day_w,
    heatwave_w,
    gwl_w,
]))

## 3 · Resolve selections

Snapshot widget values into plain Python so the rest of the notebook is
deterministic when re-run end-to-end.

In [ ]:
county: str = county_w.value
hot_day_threshold_F: int = int(hot_day_w.value)
heatwave_min_days: int = int(heatwave_w.value)
warming_levels: list[float] = list(gwl_w.value) or [1.5, 2.0]

print(f"County                    : {county}")
print(f"Hot-day threshold         : >{hot_day_threshold_F} °F")
print(f"Heat-wave minimum days    : {heatwave_min_days}")
print(f"Warming levels            : {warming_levels} °C")

## 4 · Fetch historical baseline (1981–2010)

WRF daily `t2max` and `t2min`, 9 km grid, clipped to the selected county,
converted to °F.  
*(Expect ~30 s – 3 min depending on data-store latency.)*

In [ ]:
def _fetch_hist(variable: str):
    """Fetch one WRF variable for the historical baseline."""
    return (
        ClimateData(verbosity=-1)
        .catalog("cadcat")
        .variable(variable)
        .activity_id("WRF")
        .institution_id("UCLA")
        .table_id("day")
        .grid_label("d02")
        .experiment_id("historical")
        .processes({
            "time_slice": ("1981-01-01", "2010-12-31"),
            "clip": county,
            "convert_units": "degF",
        })
        .get()
    )


hist_tmax = _fetch_hist("t2max")
hist_tmin = _fetch_hist("t2min")

print(f"t2max shape: {dict(hist_tmax.sizes)}")
print(f"t2min shape: {dict(hist_tmin.sizes)}")

## 5 · Fetch projections at each warming level

For each GWL, climakitae selects the ±15-yr window around each ensemble
member's crossing year, then clips to the county and converts to °F.

In [ ]:
def _fetch_gwl(variable: str, level: float):
    """Fetch one WRF variable for a global warming level window."""
    return (
        ClimateData(verbosity=-1)
        .catalog("cadcat")
        .variable(variable)
        .activity_id("WRF")
        .institution_id("UCLA")
        .table_id("day")
        .grid_label("d02")
        .experiment_id("ssp370")
        .processes({
            "warming_level": {"warming_levels": [level]},
            "clip": county,
            "convert_units": "degF",
        })
        .get()
    )


gwl_data: dict[float, dict[str, object]] = {}
for level in warming_levels:
    print(f"Fetching GWL {level} °C …")
    gwl_data[level] = {
        "tmax": _fetch_gwl("t2max", level),
        "tmin": _fetch_gwl("t2min", level),
    }

print("Done. Warming levels fetched:", list(gwl_data))

## 6 · Compute report metrics

`compute_report_metrics` accepts an ordered dict of period label →
`PeriodInputs(tmax, tmin)` and returns a tidy `DataFrame` (metrics as rows,
periods as columns) that the renderers consume directly.

In [ ]:
periods: dict[str, PeriodInputs] = {
    "Historic\n(1981-2010)": PeriodInputs(tmax=hist_tmax, tmin=hist_tmin),
}
for level in warming_levels:
    periods[f"GWL {level:g}°C"] = PeriodInputs(
        tmax=gwl_data[level]["tmax"],
        tmin=gwl_data[level]["tmin"],
    )

summary_df = compute_report_metrics(
    periods,
    hot_day_threshold_F=hot_day_threshold_F,
    heatwave_min_days=heatwave_min_days,
    return_period_years=10,
)
summary_df

## 7 · Derive headline stat-card numbers

Three numbers that surface the most-cited changes:

In [ ]:
hist_col = summary_df.columns[0]   # e.g. 'Historic\n(1981-2010)'
proj_col = summary_df.columns[-1]  # last warming level

avg_high_row = next(r for r in summary_df.index if "Average High" in r)
hw_row       = next(r for r in summary_df.index if "Heat Waves" in r)
hot_row      = next(r for r in summary_df.index if "Hot Days" in r)

delta_high = summary_df.loc[avg_high_row, proj_col] - summary_df.loc[avg_high_row, hist_col]
hist_hw    = summary_df.loc[hw_row, hist_col]
proj_hw    = summary_df.loc[hw_row, proj_col]
extra_hot  = summary_df.loc[hot_row, proj_col] - summary_df.loc[hot_row, hist_col]

hw_label = f"{proj_hw / hist_hw:.1f}×" if hist_hw > 0 else f"+{proj_hw:.0f}"

stat_items = [
    (f"+{delta_high:.1f}°F",  f"Avg summer high vs historical ({proj_col.replace(chr(10), ' ')})"),
    (hw_label,                 "Heat-wave frequency vs historical"),
    (f"+{extra_hot:.0f}",      f"Extra hot days >{hot_day_threshold_F}°F per year"),
]

for val, caption in stat_items:
    print(f"  {val:>10}  {caption}")

## 8 · Build bars-chart input

Compares the countywide 1-in-10-yr extreme heat threshold across
historical and the most-extreme projection (last warming level).

In [ ]:
extreme_row = next(r for r in summary_df.index if "1-in-" in r)

bars_df = pd.DataFrame(
    {"Historical": [summary_df.loc[extreme_row, hist_col]],
     "Projection": [summary_df.loc[extreme_row, proj_col]]},
    index=[county],
)
bars_df

## 9 · Render the composite report figure

In [ ]:
gwl_label = " & ".join(f"{g:g}°C" for g in warming_levels)

fig = build_report_figure(
    title=f"{county.removesuffix(\" County\")} — Extreme Heat Climate Report",
    subtitle=f"WRF · daily · historical (1981–2010) vs GWL {gwl_label}",
    stat_items=stat_items,
    summary_df=summary_df,
    bars_df=bars_df,
    bars_title=f"1-in-10-yr Daily Max Temperature (°F): Historical vs Projection",
)
fig

## 10 · (Optional) Export to PNG / PDF

In [ ]:
# from pathlib import Path
#
# out_dir = Path("figures")
# out_dir.mkdir(exist_ok=True)
# slug = county.lower().replace(" ", "_")
#
# fig.savefig(out_dir / f"{slug}_climate_report.png", dpi=200, bbox_inches="tight")
# fig.savefig(out_dir / f"{slug}_climate_report.pdf",          bbox_inches="tight")
# print(f"Saved to {out_dir}/")